# 🌱 Krishi Sakhi — Precision Plant & Crop Multi-Model Classifier

This Jupyter Notebook trains, benchmarks, evaluates, and exports **Machine Learning models** for precision agriculture on the comprehensive **58 Plant & Crop Varieties Dataset**.

### 🎯 Key Objectives:
1. **Diverse Botanical Scope**: Classifies 58 distinct crops across 8 categories (Cereals, Pulses, Fruits, Vegetables, Cash Crops, Plantation, Oilseeds, Spices).
2. **Multi-Parametric Agronomic Inputs**: Incorporates soil macronutrients ($N, P, K$), $pH$, ambient weather ($temperature, humidity, rainfall$), soil texture, water requirements, and crop phenology.
3. **Model Benchmarking**: Trains and compares 5 ML architectures:
   - **Random Forest Classifier**
   - **Gradient Boosting Classifier**
   - **Decision Tree Classifier**
   - **K-Nearest Neighbors (KNN)**
   - **Multinomial Logistic Regression**
4. **Validation & Feature Importance**: Evaluates cross-validation generalization and derives top agronomic drivers.
5. **Production Deployment**: Serializes the trained pipeline into `best_crop_model.joblib` for direct backend API inference.

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib

print("[OK] All core data science and machine learning packages loaded successfully.")

## 1. Load the Comprehensive Plants & Crops Dataset
The dataset contains 4,060 samples spanning 58 plant and crop classes across all agro-climatic zones.

In [ ]:
# Resolve dataset path dynamically
possible_paths = [
    os.path.join("..", "data", "comprehensive_plants_and_crops.csv"),
    os.path.join("data", "comprehensive_plants_and_crops.csv"),
    "comprehensive_plants_and_crops.csv"
]
DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate comprehensive_plants_and_crops.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset Dimensions : {df.shape[0]} samples × {df.shape[1]} features")
print(f"Total Crop Classes : {df['label'].nunique()} distinct plants/crops")
print(f"Botanical Categories: {df['category'].unique().tolist()}")

df.head(8)

## 2. Exploratory Data Analysis & Agronomic Distributions
Examine summary statistics and category distribution.

In [ ]:
# Statistical summary of numeric agronomic parameters
print("=== Descriptive Statistics ===")
display(df.describe().round(2))

# Samples per botanical category
cat_dist = df['category'].value_counts()
print("\n=== Samples per Category ===")
display(cat_dist)

## 3. Data Preprocessing & Feature Engineering
- Separate predictors ($X$) and target crop label ($y$).
- Encode categorical attributes (`soil_type`, `category`, `water_requirement`) using One-Hot Encoding.
- Stratified 80/20 train/test split.
- Standard feature scaling for distance-sensitive models.

In [ ]:
# Separate raw features and target
X_raw = df.drop(columns=['label'])
y_raw = df['label']

# Label encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = list(label_encoder.classes_)
print(f"Target classes encoded: {len(class_names)} classes.")

# One-hot encoding for categorical variables
categorical_cols = ['soil_type', 'category', 'water_requirement']
numerical_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'growth_duration_days', 'sunlight_hours']

X_encoded = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)
feature_names = list(X_encoded.columns)
print(f"Total features after encoding: {len(feature_names)}")

# Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} records")
print(f"Testing set : {X_test.shape[0]} records")

# Standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Multi-Model Training & Comparative Benchmarking
We train 5 distinct classification models and measure **Train Accuracy**, **Test Accuracy**, **Weighted Precision**, **Recall**, and **F1 Score**.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=30, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=20),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

benchmark_records = []

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train_scaled, y_train)
    
    train_pred = clf.predict(X_train_scaled)
    test_pred = clf.predict(X_test_scaled)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, test_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, test_pred, average='weighted', zero_division=0)
    
    benchmark_records.append({
        "Model": name,
        "Train Accuracy (%)": round(train_acc * 100, 2),
        "Test Accuracy (%)": round(test_acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1 Score (%)": round(f1 * 100, 2)
    })

results_df = pd.DataFrame(benchmark_records)
results_df.sort_values(by="Test Accuracy (%)", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

print("\n=== Model Performance Comparison ===")
display(results_df)

## 5. Cross-Validation & Generalization Verification
Perform 5-fold Stratified Cross-Validation on the top-ranking model.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Champion Model: {best_model_name}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

print(f"5-Fold CV Accuracy Scores: {[round(s * 100, 2) for s in cv_scores]} %")
print(f"Mean CV Accuracy: {round(cv_scores.mean() * 100, 2)}% (+/- {round(cv_scores.std() * 100, 2)}%)")

## 6. Agronomic Feature Importance
Identify which soil and climate variables exert the greatest influence on crop selection.

In [ ]:
rf = models["Random Forest"]
importances = rf.feature_importances_
feat_importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("Top 15 Most Influential Agronomic Features:")
display(feat_importance_series.head(15).round(4))

## 7. Multi-Crop Recommendation Pipeline (Inference)
Define an inference function that accepts raw farm conditions and predicts the Top-3 optimal crops with confidence scores.

In [ ]:
def predict_top_crops(farm_profile: dict, top_k: int = 3):
    input_df = pd.DataFrame([farm_profile])
    input_encoded = pd.get_dummies(input_df, columns=categorical_cols, drop_first=False)
    
    # Ensure all feature columns match training schema
    for col in feature_names:
        if col not in input_encoded.columns:
            input_encoded[col] = 0
    input_encoded = input_encoded[feature_names]
    
    # Scale input
    input_scaled = scaler.transform(input_encoded)
    
    # Predict probabilities
    probs = best_model.predict_proba(input_scaled)[0]
    top_k_indices = np.argsort(probs)[::-1][:top_k]
    
    recommendations = []
    for rank, idx in enumerate(top_k_indices, 1):
        crop = class_names[idx]
        conf = round(float(probs[idx]) * 100, 2)
        recommendations.append({
            "rank": rank,
            "crop": crop,
            "confidence": f"{conf}%"
        })
    return recommendations

# Test Scenario A: High-rainfall paddy parcel
paddy_parcel = {
    'N': 82, 'P': 46, 'K': 40, 'temperature': 24.5, 'humidity': 83.0, 'ph': 6.4, 'rainfall': 230.0,
    'soil_type': 'Clayey', 'category': 'Cereal', 'growth_duration_days': 120,
    'water_requirement': 'Very High', 'sunlight_hours': 7.0
}

# Test Scenario B: Arid sandy millet parcel
millet_parcel = {
    'N': 58, 'P': 28, 'K': 24, 'temperature': 32.0, 'humidity': 40.0, 'ph': 7.2, 'rainfall': 42.0,
    'soil_type': 'Sandy', 'category': 'Cereal', 'growth_duration_days': 85,
    'water_requirement': 'Low', 'sunlight_hours': 9.0
}

print("=== Scenario A (High Moisture) ===")
print("Predictions:", predict_top_crops(paddy_parcel))

print("\n=== Scenario B (Semi-Arid Millet) ===")
print("Predictions:", predict_top_crops(millet_parcel))

## 8. Export Model Artifacts for Backend Production
Save the best-performing model, scaler, and label encoder to disk so Krishi Sakhi's API can serve inference in real-time.

In [ ]:
export_dir = os.path.join("..", "ml_models")
if not os.path.exists(export_dir):
    export_dir = "ml_models"

artifacts = {
    "model": best_model,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "feature_names": feature_names,
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "classes": class_names,
    "metadata": {
        "model_type": best_model_name,
        "num_classes": len(class_names),
        "total_features": len(feature_names),
        "test_accuracy": float(results_df.iloc[0]["Test Accuracy (%)"])
    }
}

artifact_path = os.path.join(export_dir, "best_crop_model.joblib")
joblib.dump(artifacts, artifact_path)
print(f"[SUCCESS] Production artifacts successfully serialized to: {os.path.abspath(artifact_path)}")
print(f"   Model Type   : {best_model_name}")
print(f"   Total Classes: {len(class_names)}")